# Numbers Don't Lie - MSSQL Source

Privacy-preserving synthetic data generation using **SQL Server / Azure SQL** table statistics as input.

This notebook reads column-level statistics from SQL Server system views and DBCC SHOW_STATISTICS,
then generates synthetic data with the same statistical profile. No actual data rows are loaded.

Architecture:
1. Configuration - Connection details, source/destination, parameters
2. Statistics Interface - Shared contract (same as Delta variant)
3. MSSQL Statistics Reader - Extract metadata from SQL Server system views
4. Synthetic Data Generator - Create fake data from statistics (reused)
5. Writer - Save generated data as Delta tables (reused)
6. Orchestrator - Process multiple tables in batch

## 1. Configuration

Configure SQL Server connection, source tables, destination, and generation parameters.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Any, Tuple
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
import struct

# USER CONFIGURATION

# SQL Server connection
# Uses DefaultAzureCredential - ensure your identity has access to the database
SQL_SERVER = "your-server.database.windows.net"
SQL_DATABASE = "your-database"

# Source tables to process
# Format options:
#   - "schema.table" for a single table
#   - "schema" for all tables in a schema
#   - ["schema.table1", "schema.table2"] for specific tables
SOURCE_TABLES = "dbo"  # Process all tables in dbo schema

# Destination for generated data (Delta table in Fabric)
DESTINATION_PATH = "test_lakehouse.synthetic"

# Scale factor for generated data (1.0 = same row count as source)
SCALE_FACTOR = 1.1

# Enable V-Order optimization for destination tables
ENABLE_VORDER = True

# Create statistics with FULLSCAN if they don't exist?
# Set to True to auto-create missing statistics (requires ALTER permissions)
# Set to False to skip tables/columns without existing statistics
CREATE_STATISTICS_IF_MISSING = True

print(f"Server: {SQL_SERVER}")
print(f"Database: {SQL_DATABASE}")
print(f"Source: {SOURCE_TABLES}")
print(f"Destination: {DESTINATION_PATH}")
print(f"Scale Factor: {SCALE_FACTOR}")
print(f"V-Order: {ENABLE_VORDER}")
print(f"Create missing statistics: {CREATE_STATISTICS_IF_MISSING}")

## 2. Authentication & Connection

Establishes connection to Azure SQL using `DefaultAzureCredential` (supports managed identity, Azure CLI, VS Code, etc.).

In [ ]:
import pyodbc
from azure.identity import DefaultAzureCredential


def get_mssql_connection(server: str, database: str) -> pyodbc.Connection:
    """
    Create a connection to Azure SQL using DefaultAzureCredential.
    
    The token is acquired for the Azure SQL resource and passed via the
    ODBC driver's SQL_COPT_SS_ACCESS_TOKEN connection attribute.
    """
    credential = DefaultAzureCredential()
    token = credential.get_token("https://database.windows.net/.default")
    
    # Encode token for ODBC driver
    token_bytes = token.token.encode("UTF-16-LE")
    token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
    
    conn_str = (
        f"Driver={{ODBC Driver 18 for SQL Server}};"
        f"Server={server};"
        f"Database={database};"
        f"Encrypt=yes;"
        f"TrustServerCertificate=no;"
    )
    
    # 1256 = SQL_COPT_SS_ACCESS_TOKEN
    conn = pyodbc.connect(conn_str, attrs_before={1256: token_struct})
    print(f"Connected to {server}/{database} using DefaultAzureCredential")
    return conn


# Establish connection
conn = get_mssql_connection(SQL_SERVER, SQL_DATABASE)
print("Connection established successfully.")

## 3. Statistics Interface

Shared data classes for table and column statistics (same contract as the Delta variant).

In [ ]:
@dataclass
class ForeignKeyRelationship:
    """Describes a foreign key relationship between two tables"""
    fk_schema: str
    fk_table: str
    fk_column: str
    pk_schema: str
    pk_table: str
    pk_column: str
    constraint_name: str

    @property
    def fk_full_name(self) -> str:
        return f"{self.fk_schema}.{self.fk_table}"

    @property
    def pk_full_name(self) -> str:
        return f"{self.pk_schema}.{self.pk_table}"


@dataclass
class ColumnStats:
    """Statistics for a single column"""
    name: str
    data_type: str
    nullable: bool
    distinct_count: Optional[int] = None
    min_value: Optional[Any] = None
    max_value: Optional[Any] = None
    null_count: int = 0
    avg_length: Optional[float] = None
    max_length: Optional[int] = None
    # FK metadata: if this column references another table
    fk_reference: Optional[ForeignKeyRelationship] = None

    @property
    def null_ratio(self) -> float:
        """Calculate the ratio of null values"""
        if self.distinct_count is None:
            return 0.0
        total = self.distinct_count + self.null_count
        return self.null_count / total if total > 0 else 0.0

    @property
    def is_categorical(self) -> bool:
        """Determine if column is categorical based on cardinality"""
        if self.data_type not in ["string", "varchar", "char", "nvarchar", "nchar"]:
            return False
        if self.distinct_count is None:
            return False
        total_count = self.distinct_count + self.null_count
        return total_count > 0 and (self.distinct_count / total_count) < 0.05

    @property
    def is_foreign_key(self) -> bool:
        """Check if this column is a foreign key"""
        return self.fk_reference is not None


@dataclass
class TableStats:
    """Complete statistics for a table"""
    database_name: str
    schema_name: str
    table_name: str
    num_records: int
    columns: List[ColumnStats]
    foreign_keys: List[ForeignKeyRelationship] = None

    def __post_init__(self):
        if self.foreign_keys is None:
            self.foreign_keys = []

    @property
    def full_name(self) -> str:
        """Get fully qualified table name"""
        parts = [p for p in [self.database_name, self.schema_name, self.table_name] if p]
        return ".".join(parts)

    @property
    def schema_table_name(self) -> str:
        """Get schema.table name without database"""
        return f"{self.schema_name}.{self.table_name}"

    @property
    def parent_tables(self) -> List[str]:
        """Get list of parent table names (schema.table) this table depends on"""
        return list(set(fk.pk_full_name for fk in self.foreign_keys))

    def get_column(self, name: str) -> Optional[ColumnStats]:
        """Get column stats by name"""
        for col in self.columns:
            if col.name == name:
                return col
        return None


print("Statistics interface defined (with FK support)")

## 3b. Dependency Resolver

Determines the insertion order for tables based on foreign key relationships using topological sort.
Tables with no FK dependencies (root/primary tables) are processed first, then their dependents, and so on recursively through any depth of relationships.

In [ ]:
from collections import defaultdict, deque


class DependencyResolver:
    """Resolves table insertion order based on foreign key relationships using topological sort"""

    def resolve_order(self, tables: List[Tuple[str, str, str]], 
                      foreign_keys: List[ForeignKeyRelationship]) -> List[List[Tuple[str, str, str]]]:
        """
        Returns tables grouped into levels for insertion order.
        Level 0 = no FK dependencies (root tables), Level 1 = depends only on Level 0, etc.
        
        Tables within the same level can be processed in any order.
        Self-referencing FKs are ignored for ordering purposes.
        """
        # Build lookup: schema.table -> (database, schema, table)
        table_lookup = {}
        for db, schema, table in tables:
            key = f"{schema}.{table}"
            table_lookup[key] = (db, schema, table)

        # Build adjacency: child -> set of parents it depends on
        # Only consider FKs where both sides are in our table set
        dependencies = defaultdict(set)
        for key in table_lookup:
            dependencies[key] = set()

        for fk in foreign_keys:
            child = fk.fk_full_name
            parent = fk.pk_full_name
            # Skip self-references and tables not in our set
            if child == parent:
                continue
            if child not in table_lookup or parent not in table_lookup:
                continue
            dependencies[child].add(parent)

        # Kahn's algorithm for topological sort with level grouping
        in_degree = {key: len(deps) for key, deps in dependencies.items()}
        # Reverse map: parent -> children that depend on it
        dependents = defaultdict(set)
        for child, parents in dependencies.items():
            for parent in parents:
                dependents[parent].add(child)

        # Start with tables that have no dependencies
        current_level = [key for key, deg in in_degree.items() if deg == 0]
        levels = []

        processed = set()
        while current_level:
            level_tables = [table_lookup[key] for key in current_level if key in table_lookup]
            levels.append(level_tables)
            processed.update(current_level)

            next_level = []
            for key in current_level:
                for dependent in dependents[key]:
                    in_degree[dependent] -= 1
                    if in_degree[dependent] == 0:
                        next_level.append(dependent)
            current_level = next_level

        # Check for cycles - tables that never reached in_degree 0
        remaining = set(table_lookup.keys()) - processed
        if remaining:
            print(f"  WARNING: Circular FK dependencies detected for: {remaining}")
            print(f"  These tables will be processed last (FK constraints may fail)")
            cycle_tables = [table_lookup[key] for key in remaining]
            levels.append(cycle_tables)

        return levels


print("DependencyResolver class defined")

## 4. MSSQL Statistics Reader

Reads column-level statistics from SQL Server system views. Optionally creates statistics with FULLSCAN for columns that don't have them yet.

Privacy guarantee: No actual data rows are loaded. All information comes from:
- `sys.tables` / `sys.columns` / `sys.types` for schema
- `sys.dm_db_stats_properties` for row counts
- `DBCC SHOW_STATISTICS` for min/max/distinct/density
- `sys.stats` / `sys.stats_columns` for statistics metadata

In [ ]:
class MssqlStatsReader:
    """Read statistics from SQL Server system views and DBCC SHOW_STATISTICS"""

    # SQL Server types mapped to generic types for the generator
    TYPE_MAP = {
        "int": "int", "bigint": "long", "smallint": "int", "tinyint": "int",
        "decimal": "decimal", "numeric": "decimal", "money": "decimal", "smallmoney": "decimal",
        "float": "double", "real": "float",
        "char": "char", "varchar": "varchar", "nchar": "char", "nvarchar": "varchar",
        "text": "varchar", "ntext": "varchar",
        "date": "date", "datetime": "timestamp", "datetime2": "timestamp",
        "smalldatetime": "timestamp", "datetimeoffset": "timestamp", "time": "string",
        "bit": "boolean",
        "uniqueidentifier": "string",
        "binary": "binary", "varbinary": "binary", "image": "binary",
        "xml": "string", "sql_variant": "string",
    }

    def __init__(self, conn: pyodbc.Connection, create_stats_if_missing: bool = False):
        self.conn = conn
        self.create_stats_if_missing = create_stats_if_missing

    def discover_tables(self, source: Any) -> List[Tuple[str, str, str]]:
        """
        Discover tables from source specification.
        Returns list of (database, schema, table) tuples.
        """
        cursor = self.conn.cursor()
        tables = []

        if isinstance(source, list):
            for item in source:
                parts = item.split(".")
                if len(parts) == 2:
                    schema, table = parts
                elif len(parts) == 1:
                    schema, table = "dbo", parts[0]
                else:
                    raise ValueError(f"Invalid table reference: {item}")
                tables.append((SQL_DATABASE, schema, table))
        elif "." in source:
            parts = source.split(".")
            if len(parts) == 2:
                schema, table = parts
                tables.append((SQL_DATABASE, schema, table))
            else:
                raise ValueError(f"Invalid table reference: {source}")
        else:
            cursor.execute(
                "SELECT TABLE_SCHEMA, TABLE_NAME "
                "FROM INFORMATION_SCHEMA.TABLES "
                "WHERE TABLE_TYPE = 'BASE TABLE' AND TABLE_SCHEMA = ?",
                source
            )
            for row in cursor.fetchall():
                tables.append((SQL_DATABASE, row.TABLE_SCHEMA, row.TABLE_NAME))

        cursor.close()
        print(f"Discovered {len(tables)} table(s)")
        return tables

    def discover_foreign_keys(self, tables: List[Tuple[str, str, str]]) -> List[ForeignKeyRelationship]:
        """
        Discover all foreign key relationships between the given tables.
        Queries sys.foreign_keys, sys.foreign_key_columns, and related system views.
        """
        cursor = self.conn.cursor()
        cursor.execute(
            "SELECT "
            "    fk.name AS constraint_name, "
            "    fk_sch.name AS fk_schema, "
            "    fk_tab.name AS fk_table, "
            "    fk_col.name AS fk_column, "
            "    pk_sch.name AS pk_schema, "
            "    pk_tab.name AS pk_table, "
            "    pk_col.name AS pk_column "
            "FROM sys.foreign_keys fk "
            "INNER JOIN sys.foreign_key_columns fkc "
            "    ON fk.object_id = fkc.constraint_object_id "
            "INNER JOIN sys.tables fk_tab "
            "    ON fkc.parent_object_id = fk_tab.object_id "
            "INNER JOIN sys.schemas fk_sch "
            "    ON fk_tab.schema_id = fk_sch.schema_id "
            "INNER JOIN sys.columns fk_col "
            "    ON fkc.parent_object_id = fk_col.object_id "
            "    AND fkc.parent_column_id = fk_col.column_id "
            "INNER JOIN sys.tables pk_tab "
            "    ON fkc.referenced_object_id = pk_tab.object_id "
            "INNER JOIN sys.schemas pk_sch "
            "    ON pk_tab.schema_id = pk_sch.schema_id "
            "INNER JOIN sys.columns pk_col "
            "    ON fkc.referenced_object_id = pk_col.object_id "
            "    AND fkc.referenced_column_id = pk_col.column_id "
            "ORDER BY fk.name, fkc.constraint_column_id"
        )

        # Filter to only relationships where both tables are in our set
        table_set = set(f"{schema}.{table}" for _, schema, table in tables)
        relationships = []

        for row in cursor.fetchall():
            fk_key = f"{row.fk_schema}.{row.fk_table}"
            pk_key = f"{row.pk_schema}.{row.pk_table}"
            if fk_key in table_set and pk_key in table_set:
                relationships.append(ForeignKeyRelationship(
                    fk_schema=row.fk_schema,
                    fk_table=row.fk_table,
                    fk_column=row.fk_column,
                    pk_schema=row.pk_schema,
                    pk_table=row.pk_table,
                    pk_column=row.pk_column,
                    constraint_name=row.constraint_name,
                ))

        cursor.close()
        print(f"Discovered {len(relationships)} foreign key relationship(s)")
        return relationships

    def _get_row_count(self, schema: str, table: str) -> int:
        """
        Get approximate row count from sys.dm_db_partition_stats (no table scan).
        """
        cursor = self.conn.cursor()
        cursor.execute(
            "SELECT SUM(p.rows) AS row_count "
            "FROM sys.tables t "
            "INNER JOIN sys.schemas s ON t.schema_id = s.schema_id "
            "INNER JOIN sys.partitions p ON t.object_id = p.object_id "
            "WHERE s.name = ? AND t.name = ? AND p.index_id IN (0, 1)",
            schema, table
        )
        result = cursor.fetchone()
        cursor.close()
        return int(result.row_count) if result and result.row_count else 0

    def _get_columns(self, schema: str, table: str) -> List[dict]:
        """
        Get column metadata from INFORMATION_SCHEMA.
        """
        cursor = self.conn.cursor()
        cursor.execute(
            "SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE, "
            "       CHARACTER_MAXIMUM_LENGTH, NUMERIC_PRECISION, NUMERIC_SCALE "
            "FROM INFORMATION_SCHEMA.COLUMNS "
            "WHERE TABLE_SCHEMA = ? AND TABLE_NAME = ? "
            "ORDER BY ORDINAL_POSITION",
            schema, table
        )
        columns = []
        for row in cursor.fetchall():
            columns.append({
                "name": row.COLUMN_NAME,
                "data_type": row.DATA_TYPE,
                "nullable": row.IS_NULLABLE == "YES",
                "max_length": row.CHARACTER_MAXIMUM_LENGTH,
                "precision": row.NUMERIC_PRECISION,
                "scale": row.NUMERIC_SCALE,
            })
        cursor.close()
        return columns

    def _get_existing_stats(self, schema: str, table: str) -> Dict[str, str]:
        """
        Get mapping of column_name -> statistics_name for columns that already have statistics.
        """
        cursor = self.conn.cursor()
        cursor.execute(
            "SELECT s.name AS stat_name, c.name AS col_name "
            "FROM sys.stats s "
            "INNER JOIN sys.stats_columns sc ON s.object_id = sc.object_id AND s.stats_id = sc.stats_id "
            "INNER JOIN sys.columns c ON sc.object_id = c.object_id AND sc.column_id = c.column_id "
            "INNER JOIN sys.tables t ON s.object_id = t.object_id "
            "INNER JOIN sys.schemas sch ON t.schema_id = sch.schema_id "
            "WHERE sch.name = ? AND t.name = ? AND sc.stats_column_id = 1",
            schema, table
        )
        stats_map = {}
        for row in cursor.fetchall():
            if row.col_name not in stats_map:
                stats_map[row.col_name] = row.stat_name
        cursor.close()
        return stats_map

    def _create_statistics(self, schema: str, table: str, column: str) -> Optional[str]:
        """
        Create column statistics with FULLSCAN.
        Returns the statistics name if created, None on failure.
        """
        stat_name = f"_NDL_stat_{column}"
        cursor = self.conn.cursor()
        try:
            sql = (
                f"CREATE STATISTICS [{stat_name}] "
                f"ON [{schema}].[{table}] ([{column}]) "
                f"WITH FULLSCAN"
            )
            cursor.execute(sql)
            self.conn.commit()
            print(f"  Created statistics [{stat_name}] on [{schema}].[{table}].[{column}]")
            return stat_name
        except Exception as e:
            print(f"  Warning: Could not create statistics for [{column}]: {e}")
            return None
        finally:
            cursor.close()

    def _read_dbcc_stats(self, schema: str, table: str, stat_name: str) -> dict:
        """
        Read DBCC SHOW_STATISTICS for a given statistics object.
        Returns dict with rows, density, range_high_key, range_low_key, etc.
        """
        cursor = self.conn.cursor()
        result = {
            "rows": None,
            "distinct_count": None,
            "min_value": None,
            "max_value": None,
            "avg_length": None,
        }

        try:
            full_table = f"[{schema}].[{table}]"
            cursor.execute(f"DBCC SHOW_STATISTICS ({full_table}, [{stat_name}]) WITH STAT_HEADER")
            header = cursor.fetchone()
            if header:
                result["rows"] = int(header.Rows) if hasattr(header, "Rows") and header.Rows else None
        except Exception as e:
            print(f"  Warning: STAT_HEADER failed for {stat_name}: {e}")
        finally:
            cursor.close()

        cursor = self.conn.cursor()
        try:
            full_table = f"[{schema}].[{table}]"
            cursor.execute(f"DBCC SHOW_STATISTICS ({full_table}, [{stat_name}]) WITH DENSITY_VECTOR")
            density_row = cursor.fetchone()
            if density_row:
                density = density_row[0]
                avg_len = density_row[1]
                if density and density > 0:
                    result["distinct_count"] = int(1.0 / density)
                result["avg_length"] = float(avg_len) if avg_len else None
        except Exception as e:
            print(f"  Warning: DENSITY_VECTOR failed for {stat_name}: {e}")
        finally:
            cursor.close()

        cursor = self.conn.cursor()
        try:
            full_table = f"[{schema}].[{table}]"
            cursor.execute(f"DBCC SHOW_STATISTICS ({full_table}, [{stat_name}]) WITH HISTOGRAM")
            rows = cursor.fetchall()
            if rows:
                result["min_value"] = rows[0][0]
                result["max_value"] = rows[-1][0]
        except Exception as e:
            print(f"  Warning: HISTOGRAM failed for {stat_name}: {e}")
        finally:
            cursor.close()

        return result

    def read_table_stats(self, database: str, schema: str, table: str,
                         foreign_keys: Optional[List[ForeignKeyRelationship]] = None) -> TableStats:
        """
        Read comprehensive statistics for a SQL Server table.

        Privacy guarantee: No actual data rows are read.
        All information comes from system views and statistics objects.
        """
        print(f"\nAnalyzing [{schema}].[{table}]...")

        num_records = self._get_row_count(schema, table)
        print(f"  Approximate row count: {num_records:,}")

        columns_info = self._get_columns(schema, table)
        print(f"  Columns: {len(columns_info)}")

        existing_stats = self._get_existing_stats(schema, table)
        print(f"  Existing statistics: {len(existing_stats)} column(s) covered")

        # Build FK lookup for this table: column_name -> ForeignKeyRelationship
        fk_lookup = {}
        table_fks = []
        if foreign_keys:
            for fk in foreign_keys:
                if fk.fk_schema == schema and fk.fk_table == table:
                    fk_lookup[fk.fk_column] = fk
                    table_fks.append(fk)
            if table_fks:
                print(f"  Foreign keys: {len(table_fks)} column(s) reference other tables")

        column_stats_list = []

        for col_info in columns_info:
            col_name = col_info["name"]
            raw_type = col_info["data_type"].lower()
            mapped_type = self.TYPE_MAP.get(raw_type, "string")

            col_stat = ColumnStats(
                name=col_name,
                data_type=mapped_type,
                nullable=col_info["nullable"],
                max_length=col_info["max_length"],
                fk_reference=fk_lookup.get(col_name),
            )

            # Skip reading statistics for FK columns - their values come from parent tables
            if col_stat.is_foreign_key:
                col_stat.distinct_count = None
                column_stats_list.append(col_stat)
                continue

            stat_name = existing_stats.get(col_name)

            if stat_name is None and self.create_stats_if_missing:
                stat_name = self._create_statistics(schema, table, col_name)

            if stat_name:
                dbcc_stats = self._read_dbcc_stats(schema, table, stat_name)
                col_stat.distinct_count = dbcc_stats.get("distinct_count")
                col_stat.min_value = dbcc_stats.get("min_value")
                col_stat.max_value = dbcc_stats.get("max_value")
                col_stat.avg_length = dbcc_stats.get("avg_length")

                if col_stat.distinct_count and num_records > 0:
                    if col_info["nullable"] and col_stat.distinct_count < num_records:
                        pass  # Leave null_count as 0 to be conservative
            else:
                print(f"  No statistics available for [{col_name}], using defaults")
                col_stat.distinct_count = num_records

            column_stats_list.append(col_stat)

        return TableStats(
            database_name=database,
            schema_name=schema,
            table_name=table,
            num_records=num_records,
            columns=column_stats_list,
            foreign_keys=table_fks,
        )


print("MssqlStatsReader class defined (with FK discovery)")

## 5. Synthetic Data Generator

Generates realistic fake data based on statistical metadata. All operations are distributed using PySpark functions.

In [ ]:
class SyntheticDataGenerator:
    """Generate synthetic data from table statistics"""

    def __init__(self, spark: SparkSession, scale_factor: float = 1.0):
        self.spark = spark
        self.scale_factor = scale_factor

    def generate(self, table_stats: TableStats,
                 parent_dataframes: Optional[Dict[str, DataFrame]] = None) -> DataFrame:
        """
        Generate a synthetic DataFrame based on table statistics.
        
        For FK columns, values are sampled from the corresponding parent DataFrame
        rather than generated randomly. This ensures referential integrity.
        
        Args:
            table_stats: Statistics for the table to generate
            parent_dataframes: Dict of schema.table -> DataFrame for parent tables
                               that have already been generated
        """
        if parent_dataframes is None:
            parent_dataframes = {}

        target_rows = int(table_stats.num_records * self.scale_factor)
        print(f"  Generating {target_rows:,} synthetic rows for {table_stats.full_name}...")

        # Start with a range DataFrame
        df = self.spark.range(target_rows).select(F.col("id").alias("_row_id"))

        # Generate each column
        for col_stats in table_stats.columns:
            if col_stats.is_foreign_key:
                df = self._add_fk_column(df, col_stats, parent_dataframes, target_rows)
            else:
                df = self._add_synthetic_column(df, col_stats)

        df = df.drop("_row_id")
        return df

    def _add_fk_column(self, df: DataFrame, col_stats: ColumnStats,
                       parent_dataframes: Dict[str, DataFrame],
                       target_rows: int) -> DataFrame:
        """
        Add a foreign key column by sampling values from the parent table's DataFrame.
        Uses random sampling with replacement to pick parent key values.
        """
        fk = col_stats.fk_reference
        parent_key = fk.pk_full_name
        parent_col = fk.pk_column

        if parent_key not in parent_dataframes:
            print(f"    WARNING: Parent table {parent_key} not found in generated data.")
            print(f"    Falling back to synthetic generation for [{col_stats.name}]")
            return self._add_synthetic_column(df, col_stats)

        parent_df = parent_dataframes[parent_key]

        # Get distinct PK values from parent, add a join index
        pk_values = parent_df.select(F.col(parent_col)).distinct()
        pk_count = pk_values.count()

        if pk_count == 0:
            print(f"    WARNING: Parent table {parent_key} has no values for [{parent_col}]")
            return self._add_synthetic_column(df, col_stats)

        # Add row number to parent PKs for index-based join
        pk_indexed = pk_values.withColumn(
            "_pk_idx", F.monotonically_increasing_id()
        )
        # Repartition for deterministic numbering
        pk_indexed = pk_indexed.coalesce(1).withColumn(
            "_pk_idx", F.row_number().over(
                F.window.Window.orderBy(F.monotonically_increasing_id())
            ) - 1
        )

        # Generate random index into parent PK values for each row
        df = df.withColumn(
            "_fk_join_idx",
            F.floor(F.rand() * F.lit(pk_count)).cast("long")
        )

        # Join to get parent key values
        df = df.join(
            pk_indexed,
            df["_fk_join_idx"] == pk_indexed["_pk_idx"],
            "left"
        ).drop("_fk_join_idx", "_pk_idx")

        # Rename the parent column to the FK column name
        df = df.withColumnRenamed(parent_col, col_stats.name)

        # Apply null injection if the FK column is nullable
        if col_stats.nullable and col_stats.null_ratio > 0:
            df = df.withColumn(
                col_stats.name,
                F.when(F.rand() < col_stats.null_ratio, F.lit(None)).otherwise(F.col(col_stats.name))
            )

        print(f"    [{col_stats.name}] -> sampled from {parent_key}.[{parent_col}] ({pk_count:,} distinct values)")
        return df

    def _add_synthetic_column(self, df: DataFrame, col_stats: ColumnStats) -> DataFrame:
        """Add a single synthetic column to the DataFrame"""
        data_type_lower = col_stats.data_type.lower()

        if data_type_lower in ("int", "long", "smallint", "tinyint", "bigint"):
            synthetic_col = self._generate_integer(col_stats)
        elif data_type_lower in ("double", "float", "decimal", "numeric", "money"):
            synthetic_col = self._generate_numeric(col_stats)
        elif data_type_lower in ("varchar", "char", "nvarchar", "nchar", "string"):
            synthetic_col = self._generate_string(col_stats)
        elif data_type_lower == "date":
            synthetic_col = self._generate_date(col_stats)
        elif data_type_lower == "timestamp":
            synthetic_col = self._generate_timestamp(col_stats)
        elif data_type_lower == "boolean":
            synthetic_col = self._generate_boolean(col_stats)
        elif data_type_lower == "binary":
            synthetic_col = self._generate_binary(col_stats)
        else:
            synthetic_col = self._generate_string(col_stats)

        # Apply null injection
        if col_stats.null_ratio > 0:
            synthetic_col = F.when(F.rand() < col_stats.null_ratio, F.lit(None)).otherwise(synthetic_col)

        return df.withColumn(col_stats.name, synthetic_col)

    def _generate_integer(self, col_stats: ColumnStats):
        min_val = int(col_stats.min_value) if col_stats.min_value is not None else 0
        max_val = int(col_stats.max_value) if col_stats.max_value is not None else 1000000

        if col_stats.distinct_count and col_stats.distinct_count < 100:
            return F.floor(F.rand() * col_stats.distinct_count).cast("long") + F.lit(min_val)
        else:
            range_size = max(max_val - min_val, 1)
            return F.floor(F.rand() * range_size).cast("long") + F.lit(min_val)

    def _generate_numeric(self, col_stats: ColumnStats):
        min_val = float(col_stats.min_value) if col_stats.min_value is not None else 0.0
        max_val = float(col_stats.max_value) if col_stats.max_value is not None else 1000000.0
        range_size = max(max_val - min_val, 0.01)
        return (F.rand() * range_size + F.lit(min_val)).cast("double")

    def _generate_string(self, col_stats: ColumnStats):
        if col_stats.is_categorical and col_stats.distinct_count:
            category_id = F.floor(F.rand() * col_stats.distinct_count).cast("int")
            return F.concat(F.lit("Category_"), category_id.cast("string"))
        else:
            if any(kw in col_stats.name.lower() for kw in ("guid", "uuid", "id")):
                return F.expr("uuid()")
            else:
                target_length = int(col_stats.avg_length) if col_stats.avg_length else 20
                return F.concat(
                    F.lit("SYNTH_"),
                    F.expr("uuid()"),
                    F.lit("_"),
                    F.floor(F.rand() * 1000000).cast("string")
                ).substr(1, target_length)

    def _generate_date(self, col_stats: ColumnStats):
        if col_stats.min_value and col_stats.max_value:
            min_days = F.unix_timestamp(F.lit(str(col_stats.min_value)), "yyyy-MM-dd") / 86400
            max_days = F.unix_timestamp(F.lit(str(col_stats.max_value)), "yyyy-MM-dd") / 86400
            days_range = max_days - min_days
            random_days = (F.rand() * days_range + min_days).cast("long")
            return F.from_unixtime(random_days * 86400, "yyyy-MM-dd").cast("date")
        else:
            days_ago = F.floor(F.rand() * 1825).cast("int")
            return F.date_sub(F.current_date(), days_ago)

    def _generate_timestamp(self, col_stats: ColumnStats):
        if col_stats.min_value and col_stats.max_value:
            min_ts = F.unix_timestamp(F.lit(str(col_stats.min_value)))
            max_ts = F.unix_timestamp(F.lit(str(col_stats.max_value)))
            ts_range = max_ts - min_ts
            random_ts = (F.rand() * ts_range + min_ts).cast("long")
            return F.from_unixtime(random_ts).cast("timestamp")
        else:
            seconds_ago = F.floor(F.rand() * 157680000).cast("long")
            current_unix = F.unix_timestamp(F.current_timestamp())
            return F.from_unixtime(current_unix - seconds_ago).cast("timestamp")

    def _generate_boolean(self, col_stats: ColumnStats):
        return (F.rand() > 0.5).cast("boolean")

    def _generate_binary(self, col_stats: ColumnStats):
        return F.expr("unhex(md5(uuid()))")


print("SyntheticDataGenerator class defined (with FK support)")

## 6. Writer

Writes generated synthetic data to Delta tables with optional V-Order optimization.

In [ ]:
class DeltaTableWriter:
    """Write synthetic data to Delta tables"""
    
    def __init__(self, spark: SparkSession, enable_vorder: bool = True):
        self.spark = spark
        self.enable_vorder = enable_vorder
        
        if self.enable_vorder:
            self.spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
            print("V-Order optimization enabled")
        
        self.spark.conf.set("spark.sql.caseSensitive", "true")
    
    def write_table(self, df: DataFrame, destination_path: str, source_stats: TableStats):
        """
        Write DataFrame as Delta table to destination.
        
        Supports:
            - catalog.schema.table
            - catalog.schema (uses source table name)
            - ABFSS path
        """
        if destination_path.startswith("abfss://"):
            dest_path = destination_path.rstrip("/")
            if "/Tables/" in dest_path:
                after_tables = dest_path.split("/Tables/")[-1]
                if after_tables.count("/") == 0:
                    full_path = f"{dest_path}/{source_stats.table_name}"
                else:
                    full_path = dest_path
            else:
                full_path = dest_path
            
            print(f"  Writing to ABFSS path: {full_path}...")
            (df.write
             .format("delta")
             .mode("overwrite")
             .option("delta.columnMapping.mode", "name")
             .option("delta.minReaderVersion", "2")
             .option("delta.minWriterVersion", "5")
             .option("overwriteSchema", "true")
             .save(full_path))
            
            final_count = self.spark.read.format("delta").load(full_path).count()
            print(f"  Successfully wrote {final_count:,} rows to {full_path}")
            return
        
        # Dot-separated catalog path
        parts = destination_path.split(".")
        if len(parts) == 3:
            full_name = destination_path
        elif len(parts) == 2:
            full_name = f"{destination_path}.{source_stats.table_name}"
        else:
            raise ValueError(f"Invalid destination path: {destination_path}")
        
        print(f"  Writing to {full_name}...")
        
        ordered_columns = [col_stat.name for col_stat in source_stats.columns]
        df = df.select(*ordered_columns)
        
        try:
            table_exists = self.spark.catalog.tableExists(full_name)
            
            (df.write
             .format("delta")
             .mode("overwrite")
             .option("delta.columnMapping.mode", "name")
             .option("delta.minReaderVersion", "2")
             .option("delta.minWriterVersion", "5")
             .option("overwriteSchema", "true")
             .saveAsTable(full_name))
            
            final_count = self.spark.sql(f"SELECT COUNT(*) as cnt FROM {full_name}").collect()[0].cnt
            print(f"  Successfully wrote {final_count:,} rows to {full_name}")
            
        except Exception as e:
            print(f"  Error writing table {full_name}: {e}")
            raise


print("DeltaTableWriter class defined")

## 7. Orchestrator

Coordinates the end-to-end pipeline with FK-aware insertion order:
1. Discover tables in MSSQL
2. Discover foreign key relationships between tables
3. Topological sort to determine insertion order (parents before children)
4. Process level by level: generate parent tables first, then use their key values for child FK columns
5. Write to Delta

This ensures referential integrity is maintained at any depth of FK relationships (primary -> secondary -> tertiary -> ...).

In [ ]:
class MssqlSyntheticOrchestrator:
    """Orchestrates synthetic data generation from MSSQL source with FK-aware insertion order"""

    def __init__(self,
                 spark: SparkSession,
                 conn: pyodbc.Connection,
                 source_tables: Any,
                 destination_path: str,
                 scale_factor: float = 1.0,
                 enable_vorder: bool = True,
                 create_stats_if_missing: bool = False):
        self.spark = spark
        self.conn = conn
        self.source_tables = source_tables
        self.destination_path = destination_path
        self.scale_factor = scale_factor

        self.stats_reader = MssqlStatsReader(conn, create_stats_if_missing)
        self.generator = SyntheticDataGenerator(spark, scale_factor)
        self.writer = DeltaTableWriter(spark, enable_vorder)
        self.dependency_resolver = DependencyResolver()

    def run(self) -> List[dict]:
        """Execute the full pipeline with FK-aware insertion order"""
        print("NUMBERS DON'T LIE - MSSQL Source Pipeline")
        print(f"Source: {self.source_tables}")
        print(f"Destination: {self.destination_path}")
        print(f"Scale Factor: {self.scale_factor}")

        # Step 1: Discover tables
        print("\nStep 1: Discovering tables...")
        tables = self.stats_reader.discover_tables(self.source_tables)

        if not tables:
            print("No tables found. Check your source configuration.")
            return []

        # Step 2: Discover foreign key relationships
        print("\nStep 2: Discovering foreign key relationships...")
        all_foreign_keys = self.stats_reader.discover_foreign_keys(tables)

        # Step 3: Resolve insertion order via topological sort
        print("\nStep 3: Resolving insertion order...")
        levels = self.dependency_resolver.resolve_order(tables, all_foreign_keys)

        for i, level in enumerate(levels):
            table_names = [f"[{s}].[{t}]" for _, s, t in level]
            print(f"  Level {i}: {', '.join(table_names)}")

        # Step 4: Process tables level by level
        # Keep generated DataFrames in memory so child tables can sample FK values
        generated_dataframes: Dict[str, DataFrame] = {}
        results = []
        table_counter = 0
        total_tables = sum(len(level) for level in levels)

        for level_idx, level in enumerate(levels):
            print(f"\n{'='*60}")
            print(f"Processing Level {level_idx} ({len(level)} table(s))")
            print(f"{'='*60}")

            for database, schema, table in level:
                table_counter += 1
                table_key = f"{schema}.{table}"
                print(f"\n--- Table {table_counter}/{total_tables}: [{schema}].[{table}] (Level {level_idx}) ---")

                try:
                    # Read statistics (with FK metadata)
                    table_stats = self.stats_reader.read_table_stats(
                        database, schema, table, foreign_keys=all_foreign_keys
                    )
                    print(f"  Source: {table_stats.num_records:,} rows, {len(table_stats.columns)} columns")

                    if table_stats.num_records == 0:
                        print(f"  Skipping empty table [{schema}].[{table}]")
                        results.append({"table": table_key, "status": "SKIPPED", "reason": "empty"})
                        continue

                    # Generate synthetic data (passing parent DFs for FK resolution)
                    synthetic_df = self.generator.generate(table_stats, generated_dataframes)

                    # Cache the generated DataFrame for child tables to reference
                    generated_dataframes[table_key] = synthetic_df.cache()

                    # Write to destination
                    self.writer.write_table(synthetic_df, self.destination_path, table_stats)

                    results.append({
                        "table": table_key,
                        "status": "SUCCESS",
                        "level": level_idx,
                        "source_rows": table_stats.num_records,
                        "generated_rows": int(table_stats.num_records * self.scale_factor),
                    })

                except Exception as e:
                    print(f"  FAILED: {e}")
                    import traceback
                    traceback.print_exc()
                    results.append({"table": table_key, "status": "FAILED", "error": str(e)})

        # Step 5: Unpersist cached DataFrames
        for key, cached_df in generated_dataframes.items():
            cached_df.unpersist()

        # Step 6: Summary
        self._print_summary(results)
        return results

    def _print_summary(self, results: List[dict]):
        """Print pipeline execution summary"""
        print("\nPIPELINE SUMMARY")

        success = [r for r in results if r["status"] == "SUCCESS"]
        failed = [r for r in results if r["status"] == "FAILED"]
        skipped = [r for r in results if r["status"] == "SKIPPED"]

        print(f"Total: {len(results)} | Success: {len(success)} | Failed: {len(failed)} | Skipped: {len(skipped)}")

        if success:
            total_src = sum(r["source_rows"] for r in success)
            total_gen = sum(r["generated_rows"] for r in success)
            max_level = max(r.get("level", 0) for r in success)
            print(f"Total source rows: {total_src:,}")
            print(f"Total generated rows: {total_gen:,}")
            print(f"Dependency depth: {max_level + 1} level(s)")

        if failed:
            print("\nFailed tables:")
            for r in failed:
                print(f"  {r['table']}: {r.get('error', 'Unknown')}")


print("MssqlSyntheticOrchestrator class defined (with FK-aware ordering)")

## 8. Execute Pipeline

Run the synthetic data generation pipeline with the configured parameters.

In [ ]:
# Initialize and run
orchestrator = MssqlSyntheticOrchestrator(
    spark=spark,
    conn=conn,
    source_tables=SOURCE_TABLES,
    destination_path=DESTINATION_PATH,
    scale_factor=SCALE_FACTOR,
    enable_vorder=ENABLE_VORDER,
    create_stats_if_missing=CREATE_STATISTICS_IF_MISSING,
)

results = orchestrator.run()

## 9. Cleanup

Close the MSSQL connection.

In [ ]:
conn.close()
print("MSSQL connection closed.")